In [40]:
import torch
import timm
import time
import csv
import os
import random
import platform
import numpy as np
from PIL import Image
from torchvision import transforms

In [42]:
if torch.cuda.is_available():
    !nvidia-smi
elif torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon GPU) is available.")
    print("macOS Version:", platform.mac_ver()[0])
    print("Torch MPS support:", torch.backends.mps.is_available())
else:
    print("⚠️ No GPU detected. Using CPU.")

✅ MPS (Apple Silicon GPU) is available.
macOS Version: 15.3.2
Torch MPS support: True


In [44]:
if torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple M1/M2/M3/M4 GPU
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
device_type = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

Using device: mps


In [46]:
# ------------------------------------------------------
# Model configs: map filenames → timm model names + size
# ------------------------------------------------------
model_configs = {
    "efficientnet_b0.pth":       ("efficientnet_b0", 224),
    "efficientnet_b1.pth":       ("efficientnet_b1", 240),
    "efficientnet_b2.pth":       ("efficientnet_b2", 260),
    "efficientnet_b3.pth":       ("efficientnet_b3", 300),
    "ghostnet.pth":              ("ghostnet_100", 224),
    "mobilenet-100large.pth":    ("mobilenetv3_large_100", 224),
    "mobilenet-100small.pth":    ("mobilenetv3_small_100", 224),
    "mobilenet-75small.pth":     ("mobilenetv3_small_075", 224),  # no pretrained but still measurable
    "mobilenetv2-large.pth":     ("mobilenetv2_140", 224),
    "mobilenetv2-small.pth":     ("mobilenetv2_100", 224),
    "regnety.pth":               ("regnety_008", 224),
    "resnet18.pth":              ("resnet18", 224),
}

In [48]:
# Path to your folder
folder = "./"  # same folder

# Load a sample image
image_path = "sample2.png"   # your helmet photo
img_raw = Image.open(image_path).convert("RGB")

In [50]:
# CSV output
csv_path = "model_latency_results.csv"
csv_file = open(csv_path, "w", newline="")
writer = csv.writer(csv_file)
writer.writerow(["model", "input_size", "mean_latency_ms", "std_latency", "min_latency", "max_latency"])

70

In [52]:
# ------------------------------------------------------
# Helper: time inference
# ------------------------------------------------------
def measure_latency(model, img, trials=100, warmup=10):
    latencies = []

    # warmup
    for _ in range(warmup):
        _ = model(img)

    # timed runs
    with torch.no_grad():
        for _ in range(trials):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()

            _ = model(img)

            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.perf_counter()

            latencies.append((end - start) * 1000)

    return latencies

In [54]:
# ------------------------------------------------------
# Loop through your models
# ------------------------------------------------------
for filename, (model_name, input_size) in model_configs.items():
    print("\nTesting:", filename, "→", model_name)

    # load model
    model = timm.create_model(model_name, pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(folder + filename, map_location=device))
    model = model.to(device).eval()

    # prepare image
    transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    img = transform(img_raw).unsqueeze(0).to(device)

    # measure latency
    latencies = measure_latency(model, img)

    import numpy as np
    arr = np.array(latencies)

    print(f"Avg latency: {arr.mean():.3f} ms")

    # write to CSV
    writer.writerow([
        filename, 
        input_size, 
        arr.mean(), 
        arr.std(), 
        arr.min(), 
        arr.max()
    ])

csv_file.close()
print("\nSaved benchmark results to:", csv_path)


Testing: efficientnet_b0.pth → efficientnet_b0
Avg latency: 6.602 ms

Testing: efficientnet_b1.pth → efficientnet_b1
Avg latency: 9.707 ms

Testing: efficientnet_b2.pth → efficientnet_b2
Avg latency: 9.596 ms

Testing: efficientnet_b3.pth → efficientnet_b3
Avg latency: 11.139 ms

Testing: ghostnet.pth → ghostnet_100
Avg latency: 8.522 ms

Testing: mobilenet-100large.pth → mobilenetv3_large_100
Avg latency: 5.284 ms

Testing: mobilenet-100small.pth → mobilenetv3_small_100
Avg latency: 4.344 ms

Testing: mobilenet-75small.pth → mobilenetv3_small_075
Avg latency: 4.260 ms

Testing: mobilenetv2-large.pth → mobilenetv2_140
Avg latency: 5.066 ms

Testing: mobilenetv2-small.pth → mobilenetv2_100
Avg latency: 5.042 ms

Testing: regnety.pth → regnety_008
Avg latency: 6.612 ms

Testing: resnet18.pth → resnet18
Avg latency: 1.909 ms

Saved benchmark results to: model_latency_results.csv
